In [1]:
#export MODEL="Your model here
export MODEL="unsloth/gemma-4-26B-A4B-it-GGUF:UD-IQ4_NL_XL"

In [5]:
llama-fit-params -hf $MODEL

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
common_download_file_single_online: HEAD failed, status: 404
no remote preset found, skipping
llama_params_fit_impl: projected to use 8817 MiB of device memory vs. 4913 MiB of free device memory
llama_params_fit_impl: cannot meet free memory target of 1024 MiB, need to reduce device memory by 4928 MiB
llama_params_fit_impl: context size reduced from 262144 to 28416 -> need 4932 MiB less memory in total
llama_params_fit_impl: entire model can be fit by reducing context
llama_params_fit: successfully fit params to free device memory
llama_params_fit: fitting params to free memory took 2.66 seconds
main: printing fitted CLI arguments to stdout...
-c 28416 -ngl -1 -ot "blk\.0\.ffn_(up|down|gate|gate_up)_(ch|)exps=CPU,blk\

In [18]:
# Configurable batch sizes to test to fit context
# Though it looks like batch sizes have no effects 
# as per README 
# "Increasing this value above the value of the physical batch size may improve prompt processing performance 
# when using multiple GPUs with pipeline parallelism. Default: `2048`" 

BATCH_SIZES="1024 2048" # 2 args so we can see the diff, shoud not make a difference  
UBATCH_SIZES="64 128 256 512"
FITT="750 1024"
CMOE="99"

echo "Testing different batch/ubatch/fitt combinations for ${MODEL}..." 
  
for ub in $UBATCH_SIZES; do  
    for b in $BATCH_SIZES; do
        for ft in $FITT; do
            for cmoe in $CMOE; do
                # need to add fitt for nvidia
                OUTPUT=$(llama-fit-params -hf $MODEL -b $b -ub $ub -fitt $ft -ncmoe $cmoe 2>&1)
                
                #echo "Raw output: $OUTPUT"  # Debug line  
                # Extract context and GPU layers more robustly  
                CTX=$(echo "$OUTPUT" | grep -o '\-c [0-9]*' | awk '{print $2}')  
                NGL=$(echo "$OUTPUT" | grep -o '\-ngl -\?[0-9]*' | awk '{print $2}')  
                
                if [ ! -z "$CTX" ] && [ ! -z "$NGL" ]; then  
                    echo "ub: $ub, b: $b, fitt: $ft, ncmoe: $cmoe, ctx: $CTX, ngl: $NGL"  
                else  
                    echo "No valid parameters found"  
                fi
            done
        done
    done  
done 

Testing different batch/ubatch/fitt combinations for unsloth/gemma-4-26B-A4B-it-GGUF:UD-IQ4_NL_XL...
ub: 64, b: 1024, fitt: 750, ncmoe: 99, ctx: 64000, ngl: -1
ub: 64, b: 1024, fitt: 1024, ncmoe: 99, ctx: 50432, ngl: -1
ub: 64, b: 2048, fitt: 750, ncmoe: 99, ctx: 64512, ngl: -1
ub: 64, b: 2048, fitt: 1024, ncmoe: 99, ctx: 50432, ngl: -1
ub: 128, b: 1024, fitt: 750, ncmoe: 99, ctx: 63744, ngl: -1
ub: 128, b: 1024, fitt: 1024, ncmoe: 99, ctx: 49664, ngl: -1
ub: 128, b: 2048, fitt: 750, ncmoe: 99, ctx: 63744, ngl: -1
ub: 128, b: 2048, fitt: 1024, ncmoe: 99, ctx: 49664, ngl: -1
ub: 256, b: 1024, fitt: 750, ncmoe: 99, ctx: 61184, ngl: -1
ub: 256, b: 1024, fitt: 1024, ncmoe: 99, ctx: 47616, ngl: -1
ub: 256, b: 2048, fitt: 750, ncmoe: 99, ctx: 61184, ngl: -1
ub: 256, b: 2048, fitt: 1024, ncmoe: 99, ctx: 47616, ngl: -1
ub: 512, b: 1024, fitt: 750, ncmoe: 99, ctx: 45824, ngl: -1
ub: 512, b: 1024, fitt: 1024, ncmoe: 99, ctx: 32768, ngl: -1
ub: 512, b: 2048, fitt: 750, ncmoe: 99, ctx: 45824, ngl:

In [23]:
echo "Staring llama-bench for ${MODEL}..."
llama-bench -hf $MODEL -ub 64,128,256 --mmap 0 -ngl 99 -ncmoe 99 -p 1000,2000 -n 256,512
# does ub affect pp? yes, higher ub, higher PP at the cost of lower context

Staring llama-bench for unsloth/gemma-4-26B-A4B-it-GGUF:UD-IQ4_NL_XL...
ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
| model                          |       size |     params | backend    | ngl |  n_cpu_moe | n_ubatch | mmap |            test |                  t/s |
| ------------------------------ | ---------: | ---------: | ---------- | --: | ---------: | -------: | ---: | --------------: | -------------------: |
| gemma4 26B.A4B IQ4_XS - 4.25 bpw |  13.54 GiB |    25.23 B | CUDA       |  99 |         99 |       64 |    0 |          pp1000 |        110.90 ± 0.74 |
| gemma4 26B.A4B IQ4_XS - 4.25 bpw |  13.54 GiB |    25.23 B | CUDA       |  99 |         99 |       64 |    0 |          pp2000 |        105.83 ± 1.65 |
| gemma4 26B.A4B IQ4_XS - 4.25 bpw |  13.54

In [24]:
echo "Staring llama-bench for ${MODEL}..."
llama-bench -hf $MODEL -ub 128,256 -fitt 750 --mmap 0 -ngl 99 -ncmoe 99 -p 1000,2000 -n 256,512
# and looks like -fitt 750 results in less memory: 12.8/15.5 GiB used to be 14.5/15.5 GiB

Staring llama-bench for unsloth/gemma-4-26B-A4B-it-GGUF:UD-IQ4_NL_XL...
ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
| model                          |       size |     params | backend    | ngl |  n_cpu_moe | n_ubatch | mmap |       fitt |            test |                  t/s |
| ------------------------------ | ---------: | ---------: | ---------- | --: | ---------: | -------: | ---: | ---------: | --------------: | -------------------: |
| gemma4 26B.A4B IQ4_XS - 4.25 bpw |  13.54 GiB |    25.23 B | CUDA       |  99 |         99 |      128 |    0 |        750 |          pp1000 |        183.29 ± 1.24 |
| gemma4 26B.A4B IQ4_XS - 4.25 bpw |  13.54 GiB |    25.23 B | CUDA       |  99 |         99 |      128 |    0 |        750 |          pp2000 |        175.53 

In [12]:
echo "Staring llama-bench for ${MODEL}..." 
llama-bench -hf $MODEL -ub 64 -ngl 0,20,99 -p 1000,2000 -n 256,512
# no point doing this bench with ncmoe defaulted to 0

Staring llama-bench for unsloth/gemma-4-26B-A4B-it-GGUF:UD-IQ4_NL_XL...
ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
| model                          |       size |     params | backend    | ngl | n_ubatch |            test |                  t/s |
| ------------------------------ | ---------: | ---------: | ---------- | --: | -------: | --------------: | -------------------: |
| gemma4 26B.A4B IQ4_XS - 4.25 bpw |  13.54 GiB |    25.23 B | CUDA       |   0 |       64 |          pp1000 |         50.98 ± 2.62 |
| gemma4 26B.A4B IQ4_XS - 4.25 bpw |  13.54 GiB |    25.23 B | CUDA       |   0 |       64 |          pp2000 |         51.45 ± 1.46 |



In [16]:
echo "Staring llama-bench for ${MODEL}..."
llama-bench -hf $MODEL -ub 64 -ngl 99 -ncmoe 99 --mmap 0 -fa 0,1 -p 1000,2000 -n 256,512
# flash attention has not bearing on results, 100% CPU and GPU util

Staring llama-bench for unsloth/gemma-4-26B-A4B-it-GGUF:UD-IQ4_NL_XL...
ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
| model                          |       size |     params | backend    | ngl |  n_cpu_moe | n_ubatch | fa | mmap |            test |                  t/s |
| ------------------------------ | ---------: | ---------: | ---------- | --: | ---------: | -------: | -: | ---: | --------------: | -------------------: |
| gemma4 26B.A4B IQ4_XS - 4.25 bpw |  13.54 GiB |    25.23 B | CUDA       |  99 |         99 |       64 |  0 |    0 |          pp1000 |        110.93 ± 0.52 |
| gemma4 26B.A4B IQ4_XS - 4.25 bpw |  13.54 GiB |    25.23 B | CUDA       |  99 |         99 |       64 |  0 |    0 |          pp2000 |        106.77 ± 2.34 |
| gemma4 26B.A4B IQ4_XS

In [17]:
echo "Staring llama-bench for ${MODEL}..."
llama-bench -hf $MODEL -ub 64 -ngl 99 -ncmoe 99 --mmap 0,1 -p 1000,2000 -n 256,512
# how does mmap affect, quite a lot, when disabled, PP much faster and memory fully utilized. 
# Why --mmap does not affect TG?

Staring llama-bench for unsloth/gemma-4-26B-A4B-it-GGUF:UD-IQ4_NL_XL...
ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
| model                          |       size |     params | backend    | ngl |  n_cpu_moe | n_ubatch | mmap |            test |                  t/s |
| ------------------------------ | ---------: | ---------: | ---------- | --: | ---------: | -------: | ---: | --------------: | -------------------: |
| gemma4 26B.A4B IQ4_XS - 4.25 bpw |  13.54 GiB |    25.23 B | CUDA       |  99 |         99 |       64 |    0 |          pp1000 |        110.54 ± 0.85 |
| gemma4 26B.A4B IQ4_XS - 4.25 bpw |  13.54 GiB |    25.23 B | CUDA       |  99 |         99 |       64 |    0 |          pp2000 |        106.56 ± 1.50 |
| gemma4 26B.A4B IQ4_XS - 4.25 bpw |  13.54

In [19]:
echo "Staring llama-bench for ${MODEL}..."
llama-bench -hf $MODEL -ub 64 -ngl 99 -ncmoe 99 --mmap 0 -p 1000,2000 -n 256,512
# how does fitt affect, 512 seems risky, already have a few times CBBs

Staring llama-bench for unsloth/gemma-4-26B-A4B-it-GGUF:UD-IQ4_NL_XL...
ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
error: invalid parameter for argument: --fitt
usage: llama-bench [options]

options:
  -h, --help
  --numa <distribute|isolate|numactl>         numa mode (default: disabled)
  -r, --repetitions <n>                       number of times to repeat each test (default: 5)
  --prio <-1|0|1|2|3>                         process/thread priority (default: 0)
  --delay <0...N> (seconds)                   delay between each test (default: 0)
  -o, --output <csv|json|jsonl|md|sql>        output format printed to stdout (default: md)
  -oe, --output-err <csv|json|jsonl|md|sql>   output format printed to stderr (default: none)
  --list-devices                  

: 1

In [ ]:
echo "Staring llama-cli for ${MODEL}..." 
llama-cli -lv 3 -hf $MODEL -ub 64 -fitt 256 -ngl -1 --no-warmup --no-mmproj --reasoning off --single-turn --prompt "Who are you?"

In [ ]:
echo "Staring llama-server for ${MODEL}..." 
llama-server -hf $MODEL --threads-http 1 -ub 64 -fitt 256 -ngl -1 --no-warmup --no-mmproj --reasoning off -np 1